In [4]:
import warnings
warnings.filterwarnings('ignore')

import os
import io
import time
import json
import shutil
import zipfile
import requests

import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.ticker import MaxNLocator
from rich.progress import track
from pathlib import Path

from scipy.signal import find_peaks
from scipy.stats import kurtosis, skew, pearsonr
from scipy.signal.windows import blackman

from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.preprocessing import PowerTransformer, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, StackingClassifier, VotingClassifier


In [5]:
# FD - частотная область, TD - временна́я область

cols = []
common_cols = ["Median", "Mean", "Std", "Ptp", "Max", "Min", "P90", "P75", "P25", 
               "L1", "Kurtosis", "Skewness", "Integral", "RMS", "IntegralSq", 
               "SumSq", "CumsumFinal", "CumsumMax", "CumsumMin", "CumsumRange", 
               "Variation", "OutlierCount", "PeakCount"]

frequency_cols = ["EnergySpectralCentroid", "EnergySpectralBandwidth", 
                  "MeanAmpAbove10Hz", "StdAmpAbove10Hz", "EnergyAbove10Hz"]

for channel in ["mag_g", "mag_a"]:
    for dif in ["", "dif_", "dif_dif_"]:
        for col_name in common_cols:
            cols.append(f"{channel}_TD_{dif}{col_name}")

for channel in ["gx", "gy", "gz", "ax", "ay", "az"]:
    for dif in ["", "dif_", "dif_dif_"]:
        for col_name in common_cols:
            cols.append(f"{channel}_TD_{dif}{col_name}")
        for col_name in common_cols + frequency_cols:
            cols.append(f"{channel}_FD_{dif}{col_name}")

num_of_features = len(cols)

times = {i: 0 for i in cols}
cols_without_class = cols.copy()

cols.append("Класс")
features_df_orig = pd.DataFrame(columns=cols)

print(f"Количество признаков: {num_of_features}")


Количество признаков: 1056


In [6]:
def feature_extraction(seg, c, features, frequencies=None, more10hz_idx=None):
   
    t = time.perf_counter()
    square_seg = seg**2
    tt = time.perf_counter()
    times[f"{c}RMS"] += tt - t
    times[f"{c}IntegralSq"] += tt - t
    if frequencies is not None:
        times[f"{c}EnergySpectralCentroid"] += tt - t
        times[f"{c}EnergySpectralBandwidth"] += tt - t  

    
    t = time.perf_counter()
    sum_square_seg = np.sum(square_seg)
    tt = time.perf_counter()
    times[f"{c}SumSq"] += tt - t
    if frequencies is not None:
        times[f"{c}EnergySpectralCentroid"] += tt - t
        times[f"{c}EnergySpectralBandwidth"] += tt - t

    
    t = time.perf_counter()
    std_seg = np.std(seg)
    tt = time.perf_counter()
    times[f"{c}Std"] += tt - t
    times[f"{c}Variation"] += tt - t
    times[f"{c}OutlierCount"] += tt - t
    times[f"{c}PeakCount"] += tt - t

    
    t = time.perf_counter()
    mean_seg = np.mean(seg)
    tt = time.perf_counter()
    times[f"{c}Mean"] += tt - t
    times[f"{c}Variation"] += tt - t

    
    t = time.perf_counter()
    median_seg = np.median(seg)
    tt = time.perf_counter()
    times[f"{c}Median"] += tt - t
    times[f"{c}OutlierCount"] += tt - t
    times[f"{c}PeakCount"] += tt - t
    
    
    t = time.perf_counter()
    cumsum = np.cumsum(seg)
    tt = time.perf_counter()
    times[f"{c}CumsumFinal"] += tt - t
    times[f"{c}CumsumMax"] += tt - t
    times[f"{c}CumsumMin"] += tt - t
    times[f"{c}CumsumRange"] += tt - t

    
    t = time.perf_counter()
    max_cumsum = np.max(cumsum)
    tt = time.perf_counter()
    times[f"{c}CumsumMax"] += tt - t
    times[f"{c}CumsumRange"] += tt - t

    
    t = time.perf_counter()
    min_cumsum = np.min(cumsum)
    tt = time.perf_counter()
    times[f"{c}CumsumMin"] += tt - t
    times[f"{c}CumsumRange"] += tt - t


    t = time.perf_counter()
    features.append(median_seg)              # Медиана
    tt = time.perf_counter()
    times[f"{c}Median"] += tt - t


    t = time.perf_counter()
    features.append(mean_seg)                # Средняя амплитуда
    tt = time.perf_counter()
    times[f"{c}Mean"] += tt - t


    t = time.perf_counter()
    features.append(std_seg)                 # Стандартное отклонение
    tt = time.perf_counter()
    times[f"{c}Std"] += tt - t


    t = time.perf_counter()
    features.append(np.ptp(seg))             # Размах
    tt = time.perf_counter()
    times[f"{c}Ptp"] += tt - t


    t = time.perf_counter()
    features.append(np.max(seg))             # Максимальная амплитуда
    tt = time.perf_counter()
    times[f"{c}Max"] += tt - t


    t = time.perf_counter()
    features.append(np.min(seg))             # Минимальная амплитуда
    tt = time.perf_counter()
    times[f"{c}Min"] += tt - t


    t = time.perf_counter()
    features.append(np.percentile(seg, 90))  # 90-й процентиль
    tt = time.perf_counter()
    times[f"{c}P90"] += tt - t


    t = time.perf_counter()
    features.append(np.percentile(seg, 75))  # 75-й процентиль
    tt = time.perf_counter()
    times[f"{c}P75"] += tt - t


    t = time.perf_counter()
    features.append(np.percentile(seg, 25))  # 25-й процентиль
    tt = time.perf_counter()
    times[f"{c}P25"] += tt - t


    t = time.perf_counter()
    features.append(np.sum(np.abs(seg)))     # Сумма модулей
    tt = time.perf_counter()
    times[f"{c}L1"] += tt - t


    t = time.perf_counter()
    features.append(kurtosis(seg))  # Куртозис
    tt = time.perf_counter()
    times[f"{c}Kurtosis"] += tt - t


    t = time.perf_counter()
    features.append(skew(seg))      # Ассиметрия
    tt = time.perf_counter()
    times[f"{c}Skewness"] += tt - t


    t = time.perf_counter()
    features.append(np.trapz(seg))                 # Интеграл
    tt = time.perf_counter()
    times[f"{c}Integral"] += tt - t


    t = time.perf_counter()
    features.append(np.sqrt(np.mean(square_seg)))  # RMS    
    tt = time.perf_counter()
    times[f"{c}RMS"] += tt - t


    t = time.perf_counter()
    features.append(np.trapz(square_seg))          # Интеграл (квадрат)
    tt = time.perf_counter()
    times[f"{c}IntegralSq"] += tt - t


    t = time.perf_counter()
    features.append(sum_square_seg)                # Сумма квадратов
    tt = time.perf_counter()
    times[f"{c}SumSq"] += tt - t

    
    t = time.perf_counter()
    features.append(cumsum[-1])               # Чистый дрейф за окно
    tt = time.perf_counter()
    times[f"{c}CumsumFinal"] += tt - t


    t = time.perf_counter()
    features.append(max_cumsum)               # Максимум накопления
    tt = time.perf_counter()
    times[f"{c}CumsumMax"] += tt - t


    t = time.perf_counter()
    features.append(min_cumsum)               # Минимум накопления  
    tt = time.perf_counter()
    times[f"{c}CumsumMin"] += tt - t


    t = time.perf_counter()
    features.append(max_cumsum - min_cumsum)  # Размах дрейфа
    tt = time.perf_counter()
    times[f"{c}CumsumRange"] += tt - t

    
    # Коэффициент вариации
    t = time.perf_counter()
    features.append(std_seg / mean_seg if mean_seg != 0 else 0)  
    tt = time.perf_counter()
    times[f"{c}Variation"] += tt - t


    # Количество выбросов
    t = time.perf_counter()
    features.append(np.sum(np.abs(seg - median_seg) > 0.5 * std_seg))
    tt = time.perf_counter()
    times[f"{c}OutlierCount"] += tt - t


    # Количество пиков
    t = time.perf_counter()
    features.append(len(find_peaks(seg, height=median_seg + 0.5 * std_seg)[0]))
    tt = time.perf_counter()
    times[f"{c}PeakCount"] += tt - t


    if frequencies is not None:
        t = time.perf_counter()
        centroid = np.sum(frequencies * square_seg) / sum_square_seg
        tt = time.perf_counter()
        times[f"{c}EnergySpectralCentroid"] += tt - t
        times[f"{c}EnergySpectralBandwidth"] += tt - t

        
        t = time.perf_counter()
        features.append(centroid) 
        tt = time.perf_counter()
        times[f"{c}EnergySpectralCentroid"] += tt - t

        
        # Ширина
        t = time.perf_counter()
        features.append(np.sqrt(np.sum((frequencies - centroid)**2 * square_seg) / sum_square_seg))  
        tt = time.perf_counter()
        times[f"{c}EnergySpectralBandwidth"] += tt - t

        
    # Исследование частот >10 Гц
    if more10hz_idx is not None:
        t = time.perf_counter()
        amp_half_10 = seg[more10hz_idx:]
        tt = time.perf_counter()
        times[f"{c}MeanAmpAbove10Hz"] += tt - t
        times[f"{c}StdAmpAbove10Hz"] += tt - t
        times[f"{c}EnergyAbove10Hz"] += tt - t


        t = time.perf_counter()
        features.append(np.mean(amp_half_10))        # Средняя амплитуда частот >10 Гц
        tt = time.perf_counter()
        times[f"{c}MeanAmpAbove10Hz"] += tt - t
        
        
        t = time.perf_counter()
        features.append(np.std(amp_half_10))         # Стандартное отклонение частот >10 Гц
        tt = time.perf_counter()
        times[f"{c}StdAmpAbove10Hz"] += tt - t
        
        
        t = time.perf_counter()
        features.append(np.trapz(amp_half_10 ** 2))  # Энергия частот >10 Гц (квадрат)
        tt = time.perf_counter()
        times[f"{c}EnergyAbove10Hz"] += tt - t

    
    return features


Fs = 50

window_counter = 0

t_fft = 0
t_time = 0

t_mag = 0

t_diff = 0
t_diff_diff = 0

t_diff_mag = 0
t_diff_diff_mag = 0

t_fft_diff = 0
t_fft_diff_diff = 0


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


In [22]:
data_dir = 'DATA/flights'  # Путь к папкам с данными
class_folders = {
    1.0: '1.0', 
    0.9: '0.9', 
    0.8: '0.8', 
    0.7: '0.7', 
    0.6: '0.6',  
    0.5: '0.5', 
}

window_size = 64
stride = 32

window = blackman(window_size)                  # Окно Блэкмана-Наталла для уменьшения утечки спектра
freqs_0 = np.fft.rfftfreq(window_size, 1 / Fs)  # Получение только положительных частот для спектра    

more10hz_idx_0 = np.sum(freqs_0 < 10)           # Индекс для частот более 10 Гц

freqs_1 = (freqs_0[:-1] + freqs_0[1:]) / 2      # для diff_amp_half
more10hz_idx_1 = np.sum(freqs_1 < 10)           # Индекс для частот более 10 Гц

freqs_2 = (freqs_1[:-1] + freqs_1[1:]) / 2      # для diff_diff_amp_half
more10hz_idx_2 = np.sum(freqs_2 < 10)           # Индекс для частот более 10 Гц

features_df = features_df_orig.copy(deep=True)

for class_label, folder_name in class_folders.items():
    print(class_label)
    folder_path = os.path.join(data_dir, folder_name)   
    file_names = [name for name in os.listdir(folder_path) if name.endswith('.csv')]
    for file_name in file_names:            
        file_path = os.path.join(folder_path, file_name)

        df = pd.read_csv(file_path)
        data = df[['gx', 'gy', 'gz', 'ax', 'ay', 'az']].values  

        # Скользящее окно по даннымс заданным шагом
        for start in range(0, len(data) - window_size + 1, stride):
            features = []
            window_counter += 1
            segment = data[start:start + window_size]

            
            t = time.perf_counter()
            segment_windowed = segment * window[:, None]  
            yf = np.fft.rfft(segment_windowed, axis=0)    
            amplitude = np.abs(yf)      
            tt = time.perf_counter()
            t_fft += tt - t
            

            t = time.perf_counter()
            mag_g = np.sqrt(segment[:, 0]**2 + segment[:, 1]**2 + segment[:, 2]**2)
            mag_a = np.sqrt(segment[:, 3]**2 + segment[:, 4]**2 + segment[:, 5]**2)
            tt = time.perf_counter()
            t_mag += (tt - t) / 2
            

            t = time.perf_counter()
            diff_mag_g = np.diff(mag_g)
            diff_mag_a = np.diff(mag_a)
            tt = time.perf_counter()
            t_diff_mag += (tt - t) / 2
            t_diff_diff_mag += (tt - t) / 2


            t = time.perf_counter()
            diff_diff_mag_g = np.diff(diff_mag_g)
            diff_diff_mag_a = np.diff(diff_mag_a)
            tt = time.perf_counter()
            t_diff_diff_mag += (tt - t) / 2
            
                    
            for seg, c in zip([mag_g, diff_mag_g, diff_diff_mag_g, mag_a, diff_mag_a, diff_diff_mag_a], 
                              ["mag_g_TD_", "mag_g_TD_dif_", "mag_g_TD_dif_dif_", 
                               "mag_a_TD_", "mag_a_TD_dif_", "mag_a_TD_dif_dif_"]):
                
                features = feature_extraction(seg, c, features)

            # Извлечение признаков из каждой оси гироскопа и акселерометра, их производных и производных их производных
            for channel, c in zip([0, 1, 2, 3, 4, 5], ['gx', 'gy', 'gz', 'ax', 'ay', 'az']):

                
                t = time.perf_counter()
                amp_half = amplitude[:, channel] 
                tt = time.perf_counter()
                t_fft += (tt - t) / 6

                
                t = time.perf_counter()
                diff_amp_half = np.diff(amp_half)            # Первая производная амплитуд спектра
                tt = time.perf_counter()
                t_fft_diff += (tt - t) / 6
                t_fft_diff_diff += (tt - t) / 6

                
                t = time.perf_counter()
                diff_diff_amp_half = np.diff(diff_amp_half)  # Вторая производная амплитуд спектра
                tt = time.perf_counter()
                t_fft_diff_diff += (tt - t) / 6

                
                t = time.perf_counter()
                seg = segment[:, channel]
                tt = time.perf_counter()
                t_time += (tt - t) / 6

                
                t = time.perf_counter()
                diff_seg = np.diff(seg)                      # Первая производная временных данных
                tt = time.perf_counter()
                t_diff += (tt - t) / 6
                t_diff_diff += (tt - t) / 6

                
                t = time.perf_counter()
                diff_diff_seg = np.diff(diff_seg)            # Вторая производная временных данных  gz_FD_dif_CumsumMin
                tt = time.perf_counter()
                t_diff_diff += (tt - t) / 6

                
                # Обработка исходных данных и их производных
                for i, (amp_half_i, seg_i, freqs, more10hz_idx, c_time, c_fft) in enumerate([[amp_half, seg, freqs_0, more10hz_idx_0, f"{c}_TD_", f"{c}_FD_"], 
                                                                                             [diff_amp_half, diff_seg, freqs_1, more10hz_idx_1, f"{c}_TD_dif_", f"{c}_FD_dif_"], 
                                                                                             [diff_diff_amp_half, diff_diff_seg, freqs_2, more10hz_idx_2, f"{c}_TD_dif_dif_", f"{c}_FD_dif_dif_"]]):
                    
                    features = feature_extraction(seg_i, c_time, features)                            # Извлечение временных признаков
                    features = feature_extraction(amp_half_i, c_fft, features, freqs, more10hz_idx)   # Извлечение частотных признаков
            

print("Расчет времени для каждого признака")
for col in cols_without_class:
    if "FD" in col:
        times[col] += t_fft

    if "TD" in col and "mag" not in col:
        times[col] += t_time

    if "mag" in col:
        times[col] += t_mag

    if "diff_diff" in col:
        if "FD" in col:
            times[col] += t_fft_diff_diff
        elif "mag" in col:
            times[col] += t_diff_diff_mag
        elif "TD" in col:
            times[col] += t_diff_diff

    elif "diff" in col:
        if "FD" in col:
            times[col] += t_fft_diff
        elif "mag" in col:
            times[col] += t_diff_mag
        elif "TD" in col:
            times[col] += t_diff

    times[col] /= window_counter

sort_times = dict(sorted(times.items(), key=lambda x: x[1]))
for col in cols_without_class:
    sort_times[col] = round(sort_times[col] * 1_000_000, 5)

# Сохранение результатов обучения в json файл
with open('DATA/related_info/sorted_feature_extraction_time.json', 'w', encoding='utf-8') as f:
    json.dump(sort_times, f, indent=4, cls=NumpyEncoder, ensure_ascii=False)
print(f"\nРезультаты сохранены в json файл")


1.0
0.9
0.7
0.5
Расчет времени для каждого признака

Результаты сохранены в json файл
